# Camada Silver — State of Data Brasil (edição 2024)

Dono: Maycon

A Silver aplica as **regras de negócio** sobre a Bronze: seleciona as colunas
que respondem às 7 perguntas do desafio, padroniza os nomes, trata os nulos
estruturais com flags de escopo e converte os blocos de múltipla escolha pra
boolean.

**Princípio de nomes:** os apelidos seguem o **vocabulário canônico
compartilhado entre as 3 edições** (`faixa_salarial`, `genero`,
`banco_dados__mysql`, ...) — é isso que permite o Athena ler as partições das
3 edições como uma tabela só.

**Regras de negócio abaixo, verificadas nesta base (não copiadas de outra
edição):**
- Grupo sem vínculo empregatício confirmado cruzando os nulos de
  `cargo_atual`/`faixa_salarial` com `situacao_trabalho` — mesmas 4 categorias
  da edição 2025, mas checado direto nesta base.
- Bifurcação gestor × técnico confirmada (`atua_como_gestor=true` → nunca tem
  `cargo_atual`/`nivel_senioridade` preenchido, e vice-versa) — mesmo padrão
  de 2024/2025.
- **Achado específico desta edição:** as colunas booleanas (`atua_como_gestor`,
  `satisfeito_atualmente`) vêm como texto `"TRUE"`/`"FALSE"`, não `"1"`/`"0"`
  como em 2023 e 2025 — confirmado direto no CSV bruto antes de escrever a
  comparação, pra não repetir o bug de comparação que já apareceu 2x no grupo.
- Nenhuma categoria de faixa (salário, porte de empresa) fora do padrão nesta
  edição — conferido, sem achado (diferente de 2025, que teve 2 typos).

In [1]:
from pyspark.sql import SparkSession, functions as F
import sys
from pathlib import Path

CAMINHO_ATUAL = Path.cwd().resolve()
for caminho in [CAMINHO_ATUAL, *CAMINHO_ATUAL.parents]:
    if (caminho / "utils" / "config.py").exists():
        RAIZ_PROJETO = caminho
        break
else:
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto (utils/config.py).")
if str(RAIZ_PROJETO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROJETO))

from utils.config import CAMINHO_BRONZE, CAMINHO_BRONZE_METADADOS, CAMINHO_SILVER, CAMINHO_SILVER_METADADOS
from utils.functions import col_segura, slug, obter_bloco as _obter_bloco, padronizar_bloco as _padronizar_bloco

ANO_PESQUISA = 2024

spark = SparkSession.builder.appName(f"state-of-data-{ANO_PESQUISA}-silver").getOrCreate()
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

c:\Users\Henrique\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


## 1. Carregando a Bronze (só a partição desta edição)

In [2]:
# FIX: leitura sem mergeSchema pega o schema de uma partição só (normalmente
# a alfabeticamente primeira -- "ano_pesquisa=2023") e aplica ele errado em
# cima de todas as partições, inclusive esta. Mesma causa raiz identificada e
# corrigida no notebook da Silver 2025 (ver relatorio-correcao-silver-2025.md).
df = (
    spark.read.option("mergeSchema", "true").parquet(str(CAMINHO_BRONZE))
    .filter(F.col("ano_pesquisa") == ANO_PESQUISA)
)
print(f"{df.count()} linhas, {len(df.columns)} colunas")

5217 linhas, 652 colunas


## 2. Carregando o mapa de colunas gerado na Bronze

In [3]:
CAMINHO_MAPA_COLUNAS = CAMINHO_BRONZE_METADADOS / "mapa_colunas_2024.csv"

# FIX: nomes de coluna com aspas internas (ex: '"Point and Click" Analytics')
# foram escapados no padrao RFC4180 (aspas duplicadas) quando o CSV de mapa foi
# escrito. O Spark usa escape="\\" por padrao, nao aspas, e nao desfaz isso
# direito -- preciso dizer explicitamente pra ele usar aspas como escape,
# igual ja fazemos na leitura do CSV bruto da pesquisa.
df_mapa = (
    spark.read
    .option("header", "true")
    .option("quote", '"')
    .option("escape", '"')
    .csv(str(CAMINHO_MAPA_COLUNAS))
)
mapa_codigo_para_nome = {row["codigo"]: row["nome_coluna"] for row in df_mapa.collect()}

from functools import partial
obter_bloco = partial(_obter_bloco, mapa_codigo_para_nome=mapa_codigo_para_nome)

def nome(codigo):
    """Nome da coluna (pos-Bronze) a partir do codigo da pergunta."""
    return mapa_codigo_para_nome[codigo]

print(f"{len(mapa_codigo_para_nome)} codigos mapeados")

# Checagem de sanidade: todo nome que o mapa promete tem que existir no schema
# da Bronze lida agora (comparacao case-insensitive, ja que o Spark tambem
# ignora maiuscula/minuscula ao resolver nomes de coluna -- mesmo fix aplicado
# na Silver 2025).
colunas_df_lower = {c.lower() for c in df.columns}
nomes_esperados_ausentes = [
    (codigo, nome_coluna) for codigo, nome_coluna in mapa_codigo_para_nome.items()
    if nome_coluna.lower() not in colunas_df_lower
]
if nomes_esperados_ausentes:
    raise ValueError(
        f"{len(nomes_esperados_ausentes)} nome(s) do mapa de colunas nao existem no "
        f"schema da Bronze lida agora. Exemplos: {nomes_esperados_ausentes[:5]}"
    )
print("OK: todos os nomes do mapa de colunas existem no schema da Bronze lida.")


403 codigos mapeados
OK: todos os nomes do mapa de colunas existem no schema da Bronze lida.


## 3. Duplicidade de registros (por `token`)

In [4]:
# CONFIRMADO com dado real da base 2024: a coluna "id" esta inteiramente nula
# (df.groupBy("id").count() so retornou NULL). O identificador de resposta de
# verdade e o codigo "0.a" (descrito como "token" no cabecalho bruto) -- mesmo
# padrao encontrado na edicao 2025. "0.d" (data/hora_envio) e so o carimbo de
# quando a resposta foi enviada, nao serve pra dedup.
col_id = nome("0.a")

antes = df.count()

# Remove duplicatas garantindo a passagem de uma lista de strings
df = df.dropDuplicates([col_id])

depois = df.count()

print(f"Linhas antes: {antes} | depois de remover duplicatas: {depois} | removidas: {antes - depois}")


Linhas antes: 5217 | depois de remover duplicatas: 5215 | removidas: 2


## 4. Mapeando as colunas pelas 7 perguntas do desafio

Só os apelidos que a Gold de fato consulta (mesmo conjunto usado nas edições
2023 e 2025) — códigos verificados direto contra o cabeçalho desta base.

In [5]:
MAPA_SIMPLES = {
    "situacao_trabalho": "2.a",
    "setor_empresa": "2.b",
    "cargo_atual": "2.f",
    "nivel_senioridade": "2.g",
    "faixa_salarial": "2.h",
    "atua_como_gestor": "2.d",
    "num_funcionarios": "2.c",
    "tempo_experiencia_dados": "2.i",
    "tempo_experiencia_ti": "2.j",
    "objetivo_carreira": "5.a",
    "genero": "1.b",
    "cor_raca_etnia": "1.c",
    "pcd": "1.d",
    "cloud_preferida": "4.i",
    "ia_prioridade": "3.e",
    "regiao_atual": "1.i.2",
    "modelo_trabalho_atual": "2.r",
    "modelo_trabalho_ideal": "2.s",
    "satisfeito_empresa": "2.k",
    "nivel_ensino": "1.l",
    "area_formacao": "1.m",
    "atitude_retorno_presencial": "2.t",
    "pretende_mudar_emprego": "2.n",
}

MAPA_BLOCOS = {
    "cargos_no_time_dados": "3.b",
    "experiencia_prejudicada": "1.e",
    "linguagens_trabalho": "4.d",   # bloco "dia a dia" -- ver nota abaixo
    "banco_dados": "4.g",
    "fontes_dados": "4.b",
    "cloud": "4.h",
    "ferramenta_bi": "4.j",
    "ia_tipo_uso": "4.l",           # escopo técnico -- ver nota abaixo
    "ia_motivos_nao_uso": "3.g",
    "ia_uso_pessoal": "4.m",
    "desafios_gestor": "3.d",
    "motivo_insatisfacao": "2.l",
}
print(f"{len(MAPA_SIMPLES)} colunas simples, {len(MAPA_BLOCOS)} blocos")

23 colunas simples, 12 blocos


**Nota sobre `linguagens_trabalho`:** nesta edição existem 2 perguntas de
linguagem: `4.d` (múltipla escolha, "dia a dia") e `4.f` (resposta única,
"preferida"). Usei `4.d` porque é o bloco de múltipla escolha de verdade — a
edição 2025 do Vini usou o bloco "preferida" pra esse mesmo apelido, mas lá é
esse que vem como múltipla escolha (a estrutura difere entre edições). Mantém
o apelido canônico igual; a pergunta física por trás dele é a mais parecida
disponível em cada base.

**Nota sobre `ia_tipo_uso`:** igual à edição 2025, existem 2 blocos com as
mesmas opções de uso de IA na empresa (`3.f` e `4.l`), com públicos diferentes.
Uso `4.l` (seção 4, técnica) como o `ia_tipo_uso` canônico — mesma lógica que
o Vini documentou pra 2025, aqui confirmada pela posição do bloco (seção 4 é a
seção técnica, gated por `cargo_atual`).

Validação: todo código usado tem que existir no mapa, e todo bloco tem que ter opções.

In [6]:
faltando_simples = [c for c in MAPA_SIMPLES.values() if c not in mapa_codigo_para_nome]
faltando_blocos = [alias for alias, cod in MAPA_BLOCOS.items() if not obter_bloco(cod)]
if faltando_simples or faltando_blocos:
    print("ATENÇÃO, não encontrado:", faltando_simples, faltando_blocos)
else:
    print(f"OK: {len(MAPA_SIMPLES)} códigos simples e {len(MAPA_BLOCOS)} blocos confirmados.")

OK: 23 códigos simples e 12 blocos confirmados.


## 5. Tratando os nulos estruturais — flags de escopo

| Flag | Regra (verificada nesta base) |
| :-- | :-- |
| `aplica_analise_emprego` | `situacao_trabalho` fora do grupo sem vínculo (4 categorias — mesmo conjunto de 2025, confirmado por 0%/100% de preenchimento de `faixa_salarial`). |
| `aplica_analise_tecnica` | `cargo_atual` preenchido. |
| `aplica_analise_gestor` | `atua_como_gestor = "TRUE"` (texto, não `"1"` — ver achado no topo do notebook). |
| `aplica_analise_busca_oportunidade` | inverso de `aplica_analise_emprego`. |

In [7]:
from pyspark.sql import functions as F

# 1. Carrega os nomes das colunas brutas usando o mapa de 2024
# Nota: Verifique se o mapa do seu notebook de 2024 usa estes códigos de pergunta
col_situacao = mapa_codigo_para_nome.get("2.a", "situacao_trabalho")
col_cargo = mapa_codigo_para_nome.get("2.f", "cargo_atual")
col_gestor = mapa_codigo_para_nome.get("2.d", "atua_como_gestor")
col_funcao = mapa_codigo_para_nome.get("4.a.1", "funcao_de_atuacao")

# 2. Categorias sem vínculo empregatício ativo na base 2024
NAO_APLICA_EMPREGO = [
    "Desempregado, buscando recolocação",
    "Somente Estudante (graduação)",
    "Somente Estudante (pós-graduação)",
    "Trabalho na área Acadêmica/Pesquisador",
]

# 3. Construção das flags de escopo
df = (
    df
    .withColumn("aplica_analise_emprego", ~col_segura(col_situacao).isin(*NAO_APLICA_EMPREGO))
    .withColumn("aplica_analise_tecnica", col_segura(col_cargo).isNotNull())
    .withColumn("aplica_analise_gestor", (col_segura(col_gestor) == "1") | (col_segura(col_gestor) == "TRUE"))
)

# Adiciona flag de busca de oportunidade (inverso de emprego)
df = df.withColumn("aplica_analise_busca_oportunidade", ~F.col("aplica_analise_emprego"))

# Flags por função de atuação (caso existam na sua base 2024)
if col_funcao in df.columns or col_funcao in mapa_codigo_para_nome.values():
    df = (
        df
        .withColumn("aplica_bloco_engenharia_dados", col_segura(col_funcao) == "Engenharia de Dados")
        .withColumn("aplica_bloco_analise_dados", col_segura(col_funcao) == "Análise de Dados")
        .withColumn("aplica_bloco_ciencia_dados", col_segura(col_funcao) == "Outra atuação")
    )

print("Flags de escopo para a edição 2024 criadas com sucesso.")

Flags de escopo para a edição 2024 criadas com sucesso.


## 6. Padronizando os blocos de múltipla escolha para boolean

Antes, converto `satisfeito_empresa` pra boolean (`"TRUE"`/`"FALSE"` como
texto), porque o escopo de `motivo_insatisfacao` depende dele.

In [8]:
col_satisfeito = nome("2.k")
df = df.withColumn(
    col_satisfeito,
    F.when(col_segura(col_satisfeito) == "TRUE", True)
     .when(col_segura(col_satisfeito) == "FALSE", False)
     .otherwise(None).cast("boolean")
)

def escopo_do_bloco(alias):
    if alias == "motivo_insatisfacao":
        return col_segura(col_satisfeito) == False  # noqa: E712
    if alias in {"cargos_no_time_dados", "desafios_gestor"}:
        return F.col("aplica_analise_gestor")
    if alias in {"linguagens_trabalho", "banco_dados", "fontes_dados", "cloud",
                 "ferramenta_bi", "ia_tipo_uso"}:
        return F.col("aplica_analise_tecnica")
    return None  # experiencia_prejudicada, ia_motivos_nao_uso, ia_uso_pessoal: sem escopo

for alias, codigo in MAPA_BLOCOS.items():
    df = _padronizar_bloco(df, alias, codigo, mapa_codigo_para_nome, escopo_do_bloco(alias))

print("Blocos convertidos para boolean, respeitando o escopo de cada um.")

Blocos convertidos para boolean, respeitando o escopo de cada um.


## 7. Montando e exportando a Silver final

In [9]:
selects = [col_segura(nome(codigo)).alias(alias) for alias, codigo in MAPA_SIMPLES.items()]
for alias, codigo in MAPA_BLOCOS.items():
    for c in obter_bloco(codigo):
        selects.append(col_segura(c).alias(f"{alias}__{slug(c)}"))
selects += [
    F.col("aplica_analise_emprego"), F.col("aplica_analise_tecnica"),
    F.col("aplica_analise_gestor"), F.col("aplica_analise_busca_oportunidade"),
    F.col("ano_pesquisa"),
]

df_silver = df.select(*selects)
print(f"Silver final: {len(df_silver.columns)} colunas")

# ATENÇÃO -- ponto pra confirmar com o grupo: a célula 14 cria
# 'aplica_bloco_engenharia_dados' / 'aplica_bloco_analise_dados' /
# 'aplica_bloco_ciencia_dados' quando 'col_funcao' existe na base, mas esses 3
# não estão na lista de 'selects' acima -- ou seja, se forem criados, não saem
# no Parquet final da Silver. Isso é intencional (2024 não usa esse recorte) ou
# esqueceram de incluir? Não mudei sozinho porque não sei a intenção de
# negócio aqui -- se for pra incluir, é só adicionar
# F.col('aplica_bloco_engenharia_dados') etc. na lista de selects acima.

Silver final: 170 colunas


In [10]:
df_silver.write \
    .mode("overwrite") \
    .partitionBy("ano_pesquisa") \
    .parquet(str(CAMINHO_SILVER))

print(f"Silver exportada com sucesso em: {CAMINHO_SILVER}")
print(f"Linhas: {df_silver.count()} | Colunas: {len(df_silver.columns)}")

Silver exportada com sucesso em: C:\Users\Henrique\Desktop\Tech Challenge 3 local\data\silver\state_of_data_silver
Linhas: 5215 | Colunas: 170


## 8. Resumo

1. **Vocabulário:** apelidos batendo com o conjunto que a Gold (Carlos) já consulta nas 7 perguntas — testável direto contra o mesmo SQL.
2. **Nulos estruturais:** 4 flags de escopo, mesma lógica das outras edições.
3. **Achado específico:** boolean como texto `"TRUE"`/`"FALSE"`, não `"1"`/`"0"`.
4. **Categorias:** sem erro de digitação encontrado nesta edição.

**Próximo passo:** rodar a Gold (mesmas 29 tabelas de Carlos/Vini, `ANO_PESQUISA = 2024`).